# 03: Results, the final figures

A figure factory. This notebook only produces the PNGs in `reports/figures/`
and the markdown tables that go into the README. It **draws and does not
decide**: every number it shows was computed elsewhere and persisted to disk.

## Inputs (all measured, none invented)

| File | Produced by |
|---|---|
| `reports/scored_test.parquet` | the single look at test, `make pipeline` |
| `reports/policy_comparison.csv` | the same run |
| `reports/scored_calib.parquet` (`score_raw`, `p`, `isFraud`) | the same run |
| `reports/sensitivity_tornado.csv`, `drift_by_week.csv` | `make analysis` |

Every cell **fails loudly, or warns**, when its input is missing. The rule:
never publish a number that was not measured, because a visible gap beats a
plausible placeholder.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fraudq.config import CONFIG, FIGURES_DIR, PROJECT_ROOT

# --- inputs and outputs ------------------------------------------------------
# Absolute paths derived from config.py, not relative to the notebook, so that
# it does not depend on where the kernel was launched. With "../reports",
# launching from the repository root created the figures tree OUTSIDE the
# repository, silently, because mkdir(parents=True) does not complain.
REPORTS = PROJECT_ROOT / "reports"
FIGURES = FIGURES_DIR
FIGURES.mkdir(parents=True, exist_ok=True)

SCORED_TEST = REPORTS / "scored_test.parquet"
SCORED_CALIB = REPORTS / "scored_calib.parquet"
POLICY_CSV = REPORTS / "policy_comparison.csv"
TORNADO_CSV = REPORTS / "sensitivity_tornado.csv"
DRIFT_CSV = REPORTS / "drift_by_week.csv"
SUMMARY_JSON = REPORTS / "analysis_summary.json"

# LOCAL aliases of the config, for brevity in the cells below. `fraudq.config`
# exports no COSTS and no POLICY: the single source is CONFIG.
COSTS = CONFIG.cost
POLICY = CONFIG.policy

# `scored` is used by figures 4, 5 and 6. It is read here, with the other
# inputs, and not inside an intermediate cell, so that no cell silently depends
# on another having run first.
assert SCORED_TEST.exists(), f"MISSING {SCORED_TEST}: run `make pipeline`"
scored = pd.read_parquet(SCORED_TEST)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / name, bbox_inches="tight")
    # Relative to the repository root, never absolute. An executed notebook
    # stores its output, and an absolute path here would publish the home
    # directory of whoever ran it into a public repository.
    print(f"-> {(FIGURES / name).relative_to(PROJECT_ROOT)}")

## Figure 1: the headline, what each policy costs

Cost per \$1,000 for the four policies, with the saving from (3) to (4)
annotated. It is the figure at the top of the README, and its number is the
first line of it.

In [ ]:
assert POLICY_CSV.exists(), f"MISSING {POLICY_CSV}: run `make pipeline`"
comparison = pd.read_csv(POLICY_CSV, index_col=0)

labels = {
    "approve_all": "Approve\neverything",
    "single_threshold": "Single score\nthreshold",
    "topk_by_score": "Review top-K\nby score",
    "topk_by_value": "Review top-K\nby value (ours)",
}
vals = comparison["cost_per_1k"]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars = ax.bar([labels[i] for i in comparison.index], vals,
              color=["#adb5bd", "#adb5bd", "#e76f51", "#2a9d8f"])
ax.bar_label(bars, fmt="$%.2f", padding=3)
ax.set_ylabel("Expected loss per $1,000 of volume")
ax.set_title("Ranking the review queue by value, not score")
ax.set_ylim(0, vals.max() * 1.22)

# The arrow points at the gap between the two queue policies, which is the
# saving. The `$` are escaped: unescaped, the pair delimits mathtext,
# matplotlib eats the symbols and leaves "9.56per1,000" with "per" italicised.
sav = vals["topk_by_score"] - vals["topk_by_value"]
mid = (vals["topk_by_score"] + vals["topk_by_value"]) / 2
ax.annotate(f"\\${sav:.2f} per \\$1,000\nleft on the table",
            xy=(2.97, mid), xytext=(1.55, vals.max() * 0.95),
            arrowprops=dict(arrowstyle="->"), fontsize=10, fontweight="bold")
save(fig, "fig1_policy_comparison.png")

print(comparison.to_markdown(floatfmt=".2f"))

## Figure 2: calibration before and after (design.md §6.3)

The empirical evidence for the counter-cultural rule: the raw score was not a
probability, and the calibrated one is. Input: `scored_calib.parquet`.

In [ ]:
# design.md §6.2 promises the raw score, Platt and isotonic side by side. The
# three curves and the Brier/ECE table come from `fraudq.diagnostics`, which
# refits both calibrators on the same temporal holdout inside calib the pipeline
# used, so this figure and the calibrator the pipeline chose agree by
# construction.
RELIABILITY_CSV = REPORTS / "reliability_curves.csv"
CALIB_CMP_CSV = REPORTS / "calibration_comparison.csv"
assert RELIABILITY_CSV.exists(), f"MISSING {RELIABILITY_CSV}: run `make diagnostics`"

curves = pd.read_csv(RELIABILITY_CSV)
calib_cmp = pd.read_csv(CALIB_CMP_CSV).set_index("calibrator")
METHOD = {"raw": ("Raw GBDT score", "#e76f51"),
          "platt": ("Platt scaling", "#2a9d8f"),
          "isotonic": ("Isotonic regression", "#e9c46a")}

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))

# Left: the three reliability curves on one pair of axes, which is what makes
# them comparable rather than three panels the eye has to travel between.
ax = axes[0]
ax.plot([0, 1], [0, 1], "--", lw=1, color="gray", label="Perfect calibration")
for method, (label, colour) in METHOD.items():
    t = curves[curves["method"] == method]
    ax.plot(t["mean_p"], t["frac_pos"], "o-", color=colour, ms=4, label=label)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed fraud rate")
ax.set_title("Reliability on the calibration holdout", fontsize=10)
ax.legend(fontsize=8)

# Right: Brier is the number that decided, so it gets the bars. The margins are
# small and the axis says so instead of starting at zero and hiding it.
ax = axes[1]
order = ["raw", "platt", "isotonic"]
bars = ax.bar([METHOD[m][0].split()[0] for m in order],
              [calib_cmp.loc[m, "brier"] for m in order],
              color=[METHOD[m][1] for m in order])
ax.bar_label(bars, fmt="%.5f", padding=3, fontsize=9)
lo, hi = calib_cmp["brier"].min(), calib_cmp["brier"].max()
ax.set_ylim(lo - (hi - lo) * 1.2, hi + (hi - lo) * 1.6)
ax.set_ylabel("Brier score on the holdout (lower is better)")
ax.set_title("What the choice of calibrator was decided on", fontsize=10)
save(fig, "fig2_calibration_before_after.png")

print(calib_cmp.to_markdown(floatfmt=".5f"))
winner = calib_cmp["brier"].idxmin()
print(f"\nChosen by Brier on the temporal holdout: {winner}.")
print("The margins are thousandths, and the honest reading is that the raw LightGBM")
print("score was already close to calibrated. Platt improves it slightly; isotonic,")
print("the more flexible of the two, overfits the holdout and comes last.")

## Figure 3: the tornado (design.md §7.2)

`python -m fraudq.analysis` computes the sweep and this cell only draws it. An
earlier version of this cell refitted the threshold on TEST, which is exactly
what invariant 5 forbids.

In [ ]:
from fraudq.analysis import PARAM_LABELS
from fraudq.evaluate.sensitivity import plot_tornado

assert TORNADO_CSV.exists(), f"MISSING {TORNADO_CSV}: run `make analysis`"
tornado = pd.read_csv(TORNADO_CSV)
summary = json.loads(SUMMARY_JSON.read_text())

ax = plot_tornado(tornado, base_savings=summary["base_savings_per_1k"],
                  param_labels=PARAM_LABELS)
ax.set_title("How far the conclusion moves across the assumed ranges", fontsize=10)
save(ax.figure, "fig3_tornado.png")

print(tornado.to_markdown(index=False, floatfmt=".4f"))
print(f"\nSurvives the full range: {summary['conclusion_survives_range']}")
print(f"Dominant parameter:      {summary['dominant_param']}")

## Figure 4: decision regions, and where the data actually is

Left panel: the boundaries of design.md §2.2, which depend on the amount, so a
single-threshold policy is structurally wrong. Right panel: where the mass of
(p, amount) really sits in test. If the effect had come out small, this panel
would be the empirical reason why.

In [ ]:
from fraudq.policy.costs import cost_approve, cost_block, value_of_review

p_grid = np.linspace(0.001, 0.999, 400)
a_grid = np.geomspace(5, 2000, 400)
P, A = np.meshgrid(p_grid, a_grid)
region = np.where(value_of_review(P, A, COSTS) > 0, 1,
                  np.where(cost_approve(P, A, COSTS) <= cost_block(P, A, COSTS), 0, 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)
axes[0].contourf(P, A, region, levels=[-0.5, 0.5, 1.5, 2.5],
                 colors=["#2a9d8f", "#e9c46a", "#e76f51"], alpha=0.4)
axes[0].set_yscale("log")
axes[0].set_title("Decision regions (base cost assumptions)", fontsize=10)
axes[0].set_ylabel("Amount ($, log)")

h = axes[1].hexbin(scored["p"].clip(1e-3, 1 - 1e-3), scored["TransactionAmt"].clip(5, 2000),
                   yscale="log", gridsize=45, bins="log", cmap="viridis")
axes[1].set_title("Where the test data actually lives", fontsize=10)
fig.colorbar(h, ax=axes[1], label="log10(count)")
for ax in axes:
    ax.set_xlabel("Calibrated fraud probability")
save(fig, "fig4_regions_and_joint_distribution.png")

## Figure 5: the two queues are (almost) disjoint (design.md §2.4)

The overlap between the top-K by score and the top-K by value, day by day.
With a low overlap, the claim that the two sets are almost disjoint is
measured rather than asserted.

In [ ]:
from fraudq.evaluate.policies import actions_topk_by_score, actions_topk_by_value
from fraudq.policy.simulate import simulate_queue

qs = simulate_queue(scored, actions_topk_by_score, COSTS, POLICY.daily_capacity_pct)
qv = simulate_queue(scored, actions_topk_by_value, COSTS, POLICY.daily_capacity_pct)
s_set = set(np.flatnonzero((qs.actions == "review").to_numpy()))
v_set = set(np.flatnonzero((qv.actions == "review").to_numpy()))
overlap = len(s_set & v_set) / max(len(s_set | v_set), 1)
print(f"Jaccard overlap of the two queues: {overlap:.1%}")

both, only_s, only_v = s_set & v_set, s_set - v_set, v_set - s_set
fig, ax = plt.subplots(figsize=(7, 4.4))
for idx, label, color in [(only_s, "Score queue only", "#e76f51"),
                          (only_v, "Value queue only", "#2a9d8f"),
                          (both, "Both", "#666666")]:
    sub = scored.iloc[sorted(idx)]
    ax.scatter(sub["p"], sub["TransactionAmt"], s=6, alpha=0.35, label=label, color=color)
ax.set_yscale("log")
ax.legend(markerscale=3)
ax.set_xlabel("Calibrated fraud probability")
ax.set_ylabel("Amount ($, log)")
ax.set_title(f"Two review queues, {overlap:.0%} overlap", fontsize=10)
save(fig, "fig5_queue_overlap.png")

## Figure 6: drift across the test window (design.md §7.3)

Bins of 7 days, not 30. The test window lasts 27 days, so the default bin
collapses it to a single point, and one point drawn as a trend invites reading
a drift that was never measured. `python -m fraudq.analysis` computes it.

In [ ]:
from fraudq.evaluate.metrics import pr_auc

assert DRIFT_CSV.exists(), f"MISSING {DRIFT_CSV}: run `make analysis`"
perf = pd.read_csv(DRIFT_CSV)
overall = pr_auc(scored["isFraud"], scored["p"])

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(perf["week"], perf["pr_auc"], "o-", label="Weekly")
ax.axhline(overall, linestyle="--", lw=1, color="gray",
           label=f"Full window ({overall:.3f})")
# A fixed range, with room to breathe: fitted to the data the axis runs from
# 0.50 to 0.57 and a 0.07 swing in PR-AUC fills the panel, which reads as a
# drift that is not there. The scale has to let the line look flat.
ax.set_ylim(0.35, 0.75)
ax.set_xticks(perf["week"])
ax.set_xlabel("Week of the test window")
ax.set_ylabel("PR-AUC")
ax.set_title("Ranking quality week by week over the test period", fontsize=10)
ax.legend(fontsize=8)
save(fig, "fig6_performance_by_week.png")

print(perf.to_markdown(index=False, floatfmt=".4f"))
change = (perf["pr_auc"].iloc[-1] - perf["pr_auc"].iloc[0]) / perf["pr_auc"].iloc[0]
print(f"Change from the first week to the last: {change:+.1%}")
print("27 days do not support a retraining cadence: they describe the window, not a trend.")

---

# Diagnostics

Everything above describes what the policies cost. What follows describes the
model underneath them: what the score is made of, what it agrees with, and
whether the number on test can be believed.

All of it is read from the CSVs `python -m fraudq.diagnostics --learning-curve`
writes. Nothing is computed here.

In [ ]:
IMPORTANCE_CSV = REPORTS / "feature_importance.csv"
ROC_CSV = REPORTS / "roc_curve.csv"
PR_CSV = REPORTS / "pr_curve.csv"
POINTS_CSV = REPORTS / "operating_points.csv"
CORR_CSV = REPORTS / "feature_correlation.csv"
KS_CSV = REPORTS / "score_ks.csv"
PSI_CSV = REPORTS / "psi_by_week.csv"
CURVE_CSV = REPORTS / "learning_curve.csv"
BLOCKS_CSV = REPORTS / "v_null_blocks.csv"
SCORED_TRAIN = REPORTS / "scored_train.parquet"

for path in (IMPORTANCE_CSV, ROC_CSV, PR_CSV, POINTS_CSV, CORR_CSV,
             KS_CSV, PSI_CSV, CURVE_CSV, BLOCKS_CSV, SCORED_TRAIN):
    assert path.exists(), f"MISSING {path}: run `make diagnostics`"

importance = pd.read_csv(IMPORTANCE_CSV)
roc = pd.read_csv(ROC_CSV)
pr = pd.read_csv(PR_CSV)
points = pd.read_csv(POINTS_CSV).set_index("queue")
corr = pd.read_csv(CORR_CSV, index_col=0)
ks = pd.read_csv(KS_CSV)
psi = pd.read_csv(PSI_CSV, index_col=0)
curve = pd.read_csv(CURVE_CSV)
blocks = pd.read_csv(BLOCKS_CSV)

# One colour per queue, reused across figures 7 and 11 so the reader does not
# have to relearn the legend.
QUEUE_COLOUR = {"global_topk_by_score": "#6c757d",
                "daily_topk_by_score": "#e76f51",
                "daily_topk_by_value": "#2a9d8f"}
QUEUE_LABEL = {"global_topk_by_score": "Global top-K by score (reference)",
               "daily_topk_by_score": "Daily top-K by score (policy 3)",
               "daily_topk_by_value": "Daily top-K by value (policy 4)"}

## Figure 7: the ROC, and the three places a queue can sit on it

The scalar ROC-AUC averages over operating points no review queue will ever be
at. The queue lives in the far left of the curve, where capacity is 1 % of
volume, so that is where the figure zooms.

**Three markers, not two.** A ROC curve is an object about global thresholds,
and a review queue is not one: capacity renews every day, so the same total
spend has to take the best cases of each day rather than the best of the window.
That constraint alone moves a score-ranked queue off the curve before any
question of ranking. The remaining distance to the third marker is the part
that is about ranking, which is the thesis of `design.md` §2.4.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
zoom = float(points["fpr"].max() * 1.8)

# Panel A: the whole curve, with the zoom region marked.
ax = axes[0]
ax.plot(roc["fpr"], roc["tpr"], lw=1.6, color="#264653")
ax.plot([0, 1], [0, 1], ls=":", lw=1, color="gray")
ax.add_patch(plt.Rectangle((0, 0), zoom, 1.0, fill=False, ls="--", lw=1, ec="#e76f51"))
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC over the test partition", fontsize=10)

# Panel B: the operating region, which is the only part that matters here.
ax = axes[1]
ax.plot(roc["fpr"], roc["tpr"], lw=1.6, color="#264653", label="ROC")
for queue, row in points.iterrows():
    ax.plot(row["fpr"], row["tpr"], "o", ms=9, color=QUEUE_COLOUR[queue],
            label=QUEUE_LABEL[queue], zorder=3)
ax.set_xlim(0, zoom)
ax.set_ylim(0, float(points["tpr"].max() * 1.6))
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title(f"The operating region, {int(points['capacity'].iloc[0])} reviews", fontsize=10)
# Upper right, not lower right: the value-ranked marker sits low and to the
# right, which is exactly where a lower-right legend lands on top of it. The
# ROC ends well before the right edge here, so the corner is free.
ax.legend(fontsize=7.5, loc="upper right")

# Panel C: precision-recall, where the contrast is starkest.
ax = axes[2]
ax.plot(pr["recall"], pr["precision"], lw=1.6, color="#264653")
for queue, row in points.iterrows():
    ax.plot(row["tpr"], row["precision"], "o", ms=9, color=QUEUE_COLOUR[queue], zorder=3)
ax.set_xlim(0, float(points["tpr"].max() * 1.6))
ax.set_xlabel("Recall of the review queue")
ax.set_ylabel("Precision of the review queue")
ax.set_title("Precision-recall, same three points", fontsize=10)
save(fig, "fig7_roc_and_pr_curves.png")

print(points.to_markdown(floatfmt=".4f"))
by_score, by_value = points.loc["daily_topk_by_score"], points.loc["daily_topk_by_value"]
print(f"\nQueue precision: {by_score['precision']:.1%} ranking by score, "
      f"{by_value['precision']:.1%} ranking by value.")
print("Read that against the policy table above: the value-ranked queue has the WORSE")
print("queue precision and the LOWER expected loss. It is not trying to fill the queue")
print("with fraud, it is trying to fill it with cases where an analyst changes the")
print("outcome. Obvious fraud needs no analyst: the automatic rule already blocks it.")

## Figure 8: what the score is actually made of (`design.md` §5.4)

The design refused to do archaeology on the anonymised V-columns and handed
them to LightGBM whole, promising to report importance instead. This is that
report.

Gain favours high-cardinality and continuous features, which have more places
to cut, so the per-family panel on the right is the more honest reading: it
divides the total gain by how many columns were offered to earn it.

In [ ]:
TOP_SHOWN = 20
FAMILIES = r"^(V|C|D|M|card|addr|dist|id_|uid_|amt_|P_|R_|Transaction|hour)"

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))

ax = axes[0]
top = importance.head(TOP_SHOWN).iloc[::-1]
ax.barh(top["feature"], top["gain_pct"], color="#264653")
ax.set_xlabel("Share of total gain (%)")
ax.set_title(f"Top {TOP_SHOWN} features by gain", fontsize=10)
ax.tick_params(axis="y", labelsize=8)

ax = axes[1]
fam = importance.assign(
    family=importance["feature"].str.extract(FAMILIES, expand=False).fillna("other")
)
agg = (fam.groupby("family")
          .agg(columns=("feature", "size"), gain_pct=("gain_pct", "sum"))
          .sort_values("gain_pct"))
bars = ax.barh(agg.index, agg["gain_pct"], color="#2a9d8f")
ax.bar_label(bars, labels=[f"{n} cols" for n in agg["columns"]], padding=3, fontsize=8)
ax.set_xlim(0, agg["gain_pct"].max() * 1.35)
ax.set_xlabel("Share of total gain (%)")
ax.set_title("Gain by feature family, against how many columns it has", fontsize=10)
save(fig, "fig8_feature_importance.png")

used = int((importance["gain"] > 0).sum())
print(f"{used} of {len(importance)} features were split on at least once; "
      f"{len(importance) - used} were never used.")
print(f"The top 25 carry {importance.head(25)['gain_pct'].sum():.1f} % of the gain.")
v_row, c_row = agg.loc["V"], agg.loc["C"]
print(f"\nV: {v_row['columns']} columns for {v_row['gain_pct']:.1f} % of gain "
      f"({v_row['gain_pct'] / v_row['columns']:.3f} % each).")
print(f"C: {c_row['columns']} columns for {c_row['gain_pct']:.1f} % of gain "
      f"({c_row['gain_pct'] / c_row['columns']:.3f} % each).")
print("Per column the C family is worth far more, which is what justifies §5.4:")
print("the value of this project was never going to come out of the V-columns.")

## Figure 9: how much of that importance is redundant

Spearman and not Pearson: the C-columns, the D-columns and the amount have long
tails, and Pearson on those measures the outliers rather than the relationship.
Rank correlation is also the only structure a tree ensemble can exploit.

The companion number is the null-pattern grouping of the V-columns, which is
point 1 of `design.md` §5.4 and is printed below the figure.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 7.0))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.index, fontsize=7)
ax.set_title(f"Spearman correlation, top {len(corr)} features by gain", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.046, label="Rank correlation")
save(fig, "fig9_feature_correlation.png")

upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
strong = upper[upper.abs() > 0.9].sort_values(key=abs, ascending=False)
print(f"{len(strong)} pairs above 0.9 in absolute value, out of "
      f"{len(corr) * (len(corr) - 1) // 2} pairs:")
print(strong.to_string(float_format=lambda v: f"{v:.4f}") if len(strong) else "  none")

print(f"\n{int(blocks['n_columns'].sum())} V-columns fall into {len(blocks)} "
      "null-pattern blocks (design.md §5.4, point 1):")
print(blocks[["block", "n_columns", "null_rate"]].to_markdown(index=False, floatfmt=".4f"))

## Figure 10: overtraining, measured where it can be measured

**This is the only clean overtraining diagnostic in the project**, and the
reason is the split. Comparing the score on train against the score on test
would sum overtraining and genuine drift without separating them, because the
partitions are separated in time. Here the two series come from the same
expanding-window fold, so the validation window sits immediately after the
training one and drift is close to constant between them. What is left is the
gap, and the gap is overtraining.

The folds are retrained by `fraudq.diagnostics`, which does not touch
`models/train.py`: the run aborts unless the retrained folds choose the same
number of trees the persisted booster carries.

In [ ]:
from fraudq.evaluate.metrics import pr_auc

n_folds = curve["fold"].nunique()
fig, axes = plt.subplots(1, n_folds, figsize=(3.4 * n_folds, 3.6), sharey=True)
axes = np.atleast_1d(axes)

for ax, (fold, g) in zip(axes, curve.groupby("fold")):
    best = int(g["best_iteration"].iloc[0])
    ax.plot(g["iteration"], g["fit_ap"], lw=1.4, color="#e76f51", label="Fit window")
    ax.plot(g["iteration"], g["valid_ap"], lw=1.4, color="#2a9d8f", label="Validation window")
    ax.axvline(best, ls="--", lw=1, color="gray")
    at_best = g[g["iteration"] == best]
    gap = float(at_best["gap"].iloc[0])
    ax.set_title(f"Fold {fold}: gap {gap:.3f} at round {best}", fontsize=9)
    ax.set_xlabel("Boosting round")
axes[0].set_ylabel("PR-AUC")
axes[0].legend(fontsize=8, loc="lower right")
save(fig, "fig10_learning_curve.png")

at_best = (curve[curve["iteration"] == curve["best_iteration"]]
           .set_index("fold")[["best_iteration", "fit_ap", "valid_ap", "gap"]])
print(at_best.to_markdown(floatfmt=".4f"))
print(f"\nMean out-of-sample PR-AUC across folds: {at_best['valid_ap'].mean():.4f}")
print(f"Mean overtraining gap at the chosen round:  {at_best['gap'].mean():.4f}")
print(f"PR-AUC on the held-out test partition:      {pr_auc(scored['isFraud'], scored['p']):.4f}")

## Figure 11: the score distributions, and what KS can and cannot say

The classic overtraining plot, with the caveat that makes it readable here.
Splitting by class matters: the overall distributions can differ simply because
the fraud rate moved, which says nothing about the score.

Read the statistic `D`, a distance that does not grow with the sample, and not
the p-value, which at these sample sizes is ~0 for any difference at all.

In [ ]:
scored_train = pd.read_parquet(SCORED_TRAIN)
partitions = {"train": scored_train,
              "calib": pd.read_parquet(SCORED_CALIB),
              "test": scored}
PART_COLOUR = {"train": "#e76f51", "calib": "#e9c46a", "test": "#2a9d8f"}
bins = np.linspace(0, 1, 41)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
for ax, (label, want) in zip(axes, [("Fraud", 1), ("Legitimate", 0)]):
    for name, df in partitions.items():
        values = df.loc[df["isFraud"] == want, "score_raw"]
        ax.hist(values, bins=bins, density=True, histtype="step", lw=1.6,
                color=PART_COLOUR[name], label=f"{name} (n={len(values):,})")
    ax.set_yscale("log")
    ax.set_xlabel("Raw model score")
    ax.set_ylabel("Density (log)")
    ax.set_title(f"{label} transactions", fontsize=10)
    ax.legend(fontsize=8)

    subset = "fraud" if want else "legit"
    lines = [f"KS D, {subset}:"]
    for pair in ("train_vs_test", "calib_vs_test"):
        row = ks[(ks["pair"] == pair) & (ks["subset"] == subset)]
        if len(row):
            lines.append(f"  {pair.replace('_vs_', ' vs ')}: {float(row['statistic'].iloc[0]):.3f}")
    ax.text(0.97, 0.97, "\n".join(lines), transform=ax.transAxes, ha="right", va="top",
            fontsize=8, family="monospace",
            bbox=dict(boxstyle="round", fc="white", ec="lightgray"))
save(fig, "fig11_score_distributions_ks.png")

print(ks.to_markdown(index=False, floatfmt=".4g"))
fraud_tt = float(ks.query("pair == 'train_vs_test' and subset == 'fraud'")["statistic"].iloc[0])
fraud_ct = float(ks.query("pair == 'calib_vs_test' and subset == 'fraud'")["statistic"].iloc[0])
print(f"\nOn fraud, train differs from test by D = {fraud_tt:.3f}, while the two partitions")
print(f"the model never saw differ from each other by D = {fraud_ct:.3f}.")
print("Drift would move calib and test apart too, and it does not. What separates train")
print("is the model having been fitted on it, which is the same thing figure 10 measures")
print("directly and without the confound.")

## Figure 12: feature drift (`design.md` §7.3)

PSI of each top feature, week by week of the test window, against the whole of
train as the reference. The industry rule of thumb, which is a rule of thumb and
not a theorem: below 0.10 stable, 0.10 to 0.25 moderate, above 0.25 serious.

This closes the one limitation the README declared and could not measure: the
persisted scoring carries no feature values, so PSI needed the full rebuild that
`fraudq.diagnostics` does.

In [ ]:
from fraudq.evaluate.drift import PSI_THRESHOLDS

order = psi.max().sort_values(ascending=False).index
heat = psi[order]

fig, ax = plt.subplots(figsize=(11.0, 3.6))
im = ax.imshow(heat.to_numpy(), aspect="auto", cmap="YlOrRd",
               vmin=0, vmax=max(float(heat.to_numpy().max()), PSI_THRESHOLDS["moderate"]))
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=90, fontsize=7)
ax.set_yticks(range(len(heat)))
ax.set_yticklabels([f"week {w}" for w in heat.index], fontsize=8)
ax.set_title("PSI against the training distribution, by week of the test window", fontsize=10)
fig.colorbar(im, ax=ax, fraction=0.02, label="PSI")
save(fig, "fig12_psi_heatmap.png")

worst = heat.max().max()
print(f"Largest PSI anywhere in the grid: {worst:.4f}")
print(f"Features above the 'moderate' line ({PSI_THRESHOLDS['moderate']}): "
      f"{int((heat.max() > PSI_THRESHOLDS['moderate']).sum())} of {heat.shape[1]}")
print(f"Features above the 'stable' line ({PSI_THRESHOLDS['stable']}): "
      f"{int((heat.max() > PSI_THRESHOLDS['stable']).sum())} of {heat.shape[1]}")
print("\nThe inputs did not move measurably across the 27 days of test. That is a")
print("statement about a 27 day window, not a licence to skip retraining: the fraud")
print("RATE does climb over the same window (figure 6), and PSI does not see labels.")

## Figure 13: does the result survive a bigger or smaller queue? (`design.md` §2.3)

The tornado sweeps the four cost **assumptions**. This sweeps the one
**structural** parameter, and it is a different kind of question: a reader will
want to substitute their own capacity, and a result measured at a single queue
size says nothing about whether it holds at five times that.

The first row is capacity zero, and it is a reference rather than a swept point:
with no queue both policies collapse to the same automatic rule, so that number
is what a queue has to beat. Without it the rest of the table has no scale.

In [ ]:
CAPACITY_CSV = REPORTS / "capacity_sweep.csv"
assert CAPACITY_CSV.exists(), f"MISSING {CAPACITY_CSV}: run `make analysis`"

cap = pd.read_csv(CAPACITY_CSV)
no_queue = cap.iloc[0]
swept = cap.iloc[1:]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# Left: what each policy costs, against the no-queue line. The crossing is the
# point of the panel.
ax = axes[0]
ax.axhline(no_queue["cost_by_value"], ls="--", lw=1.2, color="gray",
           label=f"No queue (\${no_queue['cost_by_value']:.2f})")
ax.plot(swept["capacity_pct"] * 100, swept["cost_by_score"], "o-",
        color="#e76f51", label="Queue ranked by score")
ax.plot(swept["capacity_pct"] * 100, swept["cost_by_value"], "o-",
        color="#2a9d8f", label="Queue ranked by value")
ax.set_xscale("log")
ax.set_xticks(swept["capacity_pct"] * 100)
ax.set_xticklabels([f"{p:g}%" for p in swept["capacity_pct"] * 100])
# A log axis adds its own minor tick labels, and they land on top of the five
# explicit ones set above.
ax.minorticks_off()
ax.set_xlabel("Review capacity, as a share of daily volume")
ax.set_ylabel("Expected loss per $1,000 of volume")
ax.set_title("What a queue is worth, by how big it is", fontsize=10)
ax.legend(fontsize=8)

# Right: the saving itself, absolute and relative, which peak in different
# places and would mislead if only one were shown.
ax = axes[1]
ax.plot(swept["capacity_pct"] * 100, swept["savings_per_1k"], "o-", color="#264653",
        label="Saving per $1,000 (left)")
ax.set_xscale("log")
ax.set_xticks(swept["capacity_pct"] * 100)
ax.set_xticklabels([f"{p:g}%" for p in swept["capacity_pct"] * 100])
ax.minorticks_off()
ax.set_xlabel("Review capacity, as a share of daily volume")
ax.set_ylabel("Saving of value over score, per $1,000")
ax.axhline(0.0, color="black", lw=0.8)
twin = ax.twinx()
twin.plot(swept["capacity_pct"] * 100, swept["savings_pct"], "s--", color="#8ab17d",
          label="Saving as % of the score-ranked cost (right)")
twin.set_ylabel("Saving (%)")
twin.spines["top"].set_visible(False)
lines = ax.get_lines()[:1] + twin.get_lines()
ax.legend(lines, [ln.get_label() for ln in lines], fontsize=8, loc="lower right")
ax.set_title("Absolute and relative saving peak in different places", fontsize=10)
save(fig, "fig13_capacity_sweep.png")

print(cap.to_markdown(index=False, floatfmt=".4f"))
worse = swept[swept["cost_by_score"] >= no_queue["cost_by_score"]]
print(f"\nNo queue at all costs \${no_queue['cost_by_value']:.4f} per \$1,000.")
if len(worse):
    upto = worse["capacity_pct"].max() * 100
    print(f"A SCORE-ranked queue costs MORE than that up to {upto:g}% capacity: every")
    print("review it buys is of a case the automatic rule had already decided, so it pays")
    print("the review cost and changes nothing.")
print("A VALUE-ranked queue is ahead from the smallest capacity swept.")
print(f"Utilisation is {swept['utilization_by_value'].min():.2f} at every capacity, so the")
print("V > 0 filter never binds: the queue always finds cases worth reviewing.")

## Figure 14: calibration where the decisions are made (`design.md` §6.3)

> A model can be well calibrated on average and badly calibrated in the top 1 %,
> which is exactly where the expensive decisions are made. The average hides the
> error where it matters.

The design says that and the project had only ever reported the average. Deciles
are taken on the **raw score**, so decile 9 is "the 10 % the model finds most
suspicious" regardless of what the calibrator did to the values.

In [ ]:
from fraudq.evaluate.metrics import ece

DECILE_CSV = REPORTS / "calibration_by_decile.csv"
assert DECILE_CSV.exists(), f"MISSING {DECILE_CSV}: run `make diagnostics`"

dec = pd.read_csv(DECILE_CSV)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))

ax = axes[0]
width = 0.4
x = np.arange(len(dec))
ax.bar(x - width / 2, dec["mean_p"], width, label="Mean predicted probability", color="#264653")
ax.bar(x + width / 2, dec["frac_pos"], width, label="Observed fraud rate", color="#2a9d8f")
ax.set_xticks(x)
ax.set_xticklabels(dec["decile"])
ax.set_xlabel("Decile of the raw score (9 = most suspicious)")
ax.set_ylabel("Probability")
ax.set_title("Predicted against observed, by decile", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
bars = ax.bar(x, dec["gap"], color="#e76f51")
ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(dec["decile"])
ax.set_xlabel("Decile of the raw score")
ax.set_ylabel("|predicted - observed|")
ax.set_title("The error the aggregate hides", fontsize=10)
save(fig, "fig14_calibration_by_decile.png")

print(dec.to_markdown(index=False, floatfmt=".4f"))
overall = ece(scored["isFraud"], scored["p"])
top = dec.iloc[-1]
print(f"\nECE over the whole partition: {overall:.4f}")
print(f"Gap in the top decile:        {top['gap']:.4f}  ({top['count']:,.0f} transactions)")
print(f"Worst decile gap:             {dec['gap'].max():.4f}")
print("The review queue lives in the top decile, so that is the number the cost layer")
print("actually depends on, and it is the one an aggregate ECE never shows.")

## Closing

With the figures in `reports/figures/`, what remains is the write-up. The final
rule: every number in the README must be traceable to a cell of this notebook
or to a file in `reports/`. **None is written from memory.**